# Nutrient validation — GETM–ERSEM/BFM vs NIOZ Jetty and RWS chemistry

Compares surface nutrient, chlorophyll and oxygen output of run
`no_BP1c/spinup_01` against two observation sets held in
`/export/lv9/projects/dws/results/validation/nutrients/`:

| source | file | nature |
|---|---|---|
| **NIOZ Jetty** | `Jetty_HWseries.csv` | one fixed station (Marsdiep, Texel), high-water time series |
| **RWS** | `Chemistry_data_via_Waterinfo_RWS.csv` | many stations, irregular bottle samples |

**Model variables** — `N1p` (phosphate), `N3n` (nitrate), `N4n` (ammonium),
`N5s` (silicate), `Chla`, `O2o`, plus derived `DIN = N3n + N4n` and the DIN:DIP
ratio. All BFM nutrient pools are in **mmol m⁻³**, `Chla` in **mg m⁻³**.

**Sampling** — surface layer (`level = 10`; the repo convention is top = 10,
bottom = 0), nearest *wet* model cell, nearest model timestep.

### Things to be aware of when reading the results

* `spinup_01` is a **repeated 2015 year**, so observations are restricted to
  2015. This tests the seasonal structure, not inter-annual skill.
* `Jetty_HWseries.csv` is a **high-water** series while model output is roughly
  daily and instantaneous. The tidal phase cannot be matched, so a systematic
  offset at this station is expected and is not by itself a model error.
* Model `Chla` is a diagnostic sum over the phytoplankton groups. Observed
  chlorophyll methods differ between the two records (fluorometric vs HPLC),
  which limits how far the comparison can be pushed.
* **Run the schema-probe cell first.** No previous script in this repo reads the
  RWS export, so its exact layout is confirmed there rather than assumed, and
  the unit conversion is keyed on what it prints.

In [35]:
# kernel's working directory
import os
print(os.getcwd())

/export/lv9


In [36]:
# imports
import csv
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from matplotlib.lines import Line2D
from scipy.spatial import cKDTree
from scipy.stats import linregress, pearsonr

# Optional – only needed for the spatial error maps and the RWS coordinate fallback.
try:
    import cartopy.crs as ccrs
    import contextily as ctx
    _HAS_MAPS = True
except Exception as _e:  # pragma: no cover
    warnings.warn(f"cartopy/contextily unavailable ({_e}); spatial maps will be skipped.")
    _HAS_MAPS = False

try:
    from pyproj import Transformer
    _HAS_PYPROJ = True
except Exception:  # pragma: no cover
    _HAS_PYPROJ = False

In [37]:
# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ──────────────────────────────────────────────────────────────────────────────

POSTPROC_DIR = Path("/export/lv9/projects/dws/results/validation/nutrients/")
BASE_OUTPUT_DIR = Path("/export/lv9/projects/dws/model_output/archived_runs/")

# Output figure directory
OUT_DIR = POSTPROC_DIR / "spinup_10"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OBS_CSV = {
    "NIOZ_JETTY" : POSTPROC_DIR / "Jetty_HWseries.csv",
    "RWS" : POSTPROC_DIR / "Chemistry_data_via_Waterinfo_RWS.csv",
}

SPINUP_NAMES = ["spinup_10"]

NC_PATTERN = "dws_500m.3d.{year}*.nc"

# ── Model sampling ────────────────────────────────────────────────────────────
# spinup_01 is a repeated 2015 year (see input_scripts/interp_hotstart/
# mix_restartfiles_spinup.ipynb), so observations are restricted to 2015 and
# matched to the nearest model timestamp.
MODEL_YEAR = 2015

# Vertical index of the surface layer.  Repo convention (pelagic_validation.ipynb,
# spin-up.ipynb): level has size 11 with top = 10 and bottom = 0.
SURFACE_LEVEL = 10

# NIOZ jetty (Marsdiep, Texel).  VERIFY against the station metadata that ships
# with the Jetty dataset (NIOZ Dataverse, doi:10.25850/nioz/7b.b.5j) – the
# collocation cell prints the snap distance so a wrong position is obvious.
JETTY_LON, JETTY_LAT = 4.7891, 53.0018
JETTY_BOX_HALF = 1          # 1 -> 3x3 neighbourhood mean plotted as a band

# ── Variables ─────────────────────────────────────────────────────────────────
# BFM pelagic state variables, all in mmol m-3 except Chla (mg m-3).
VALIDATION_VARS = ["N1p", "N3n", "N4n", "N5s", "Chla", "O2o"]
# Computed after extraction on both the model and the observation side.
DERIVED_VARS = ["DIN", "DIN_DIP"]
ALL_VARS = VALIDATION_VARS + DERIVED_VARS

VAR_LABEL = {
    "N1p": "Phosphate (N1p)",
    "N3n": "Nitrate (N3n)",
    "N4n": "Ammonium (N4n)",
    "N5s": "Silicate (N5s)",
    "Chla": "Chlorophyll-a (Chla)",
    "O2o": "Oxygen (O2o)",
    "DIN": "DIN (N3n + N4n)",
    "DIN_DIP": "DIN:DIP ratio",
}

# Units *after* conversion.  Everything molar is expressed in mmol m-3, which is
# numerically identical to umol L-1.
VAR_UNITS = {
    "N1p": "mmol P m$^{-3}$",
    "N3n": "mmol N m$^{-3}$",
    "N4n": "mmol N m$^{-3}$",
    "N5s": "mmol Si m$^{-3}$",
    "Chla": "mg m$^{-3}$",
    "O2o": "mmol O$_2$ m$^{-3}$",
    "DIN": "mmol N m$^{-3}$",
    "DIN_DIP": "mol mol$^{-1}$",
}

# Plausible ranges from the BFM plotting script by J. van der Molen
# (git show 45e062d^:output_scripts/make_getm_movie_nose_transect.py).
# Used only as a sanity check in the unit-audit cell, never to filter data.
PLAUSIBLE_RANGE = {
    "N1p": (0, 6), "N3n": (0, 150), "N4n": (0, 15), "N5s": (0, 100),
    "O2o": (100, 450), "Chla": (0, 100), "DIN": (0, 165), "DIN_DIP": (0, 200),
}

# ── Model fill values (from ncdump; identical to benthic_validation_spinup) ────
LON_VALID = (-180.0, 180.0)
LAT_VALID = (-90.0, 90.0)
BATHY_FILL = -10.0          # land cells carry bathymetry == -10
DATA_FILL_THRESHOLD = -900.0  # data vars use _FillValue = -9999

# ── Observation handling ──────────────────────────────────────────────────────
RWS_MISSING_SENTINEL = 999999999.0   # DONAR "no value" code
DROP_BELOW_DETECTION = False         # False -> substitute LOD/2; True -> drop
MAX_SNAP_DISTANCE_KM = 5.0           # stations further than this are discarded
# Nearest-time matching is unbounded by construction, so a sample taken in a
# month the run does not cover would otherwise pair with the closest available
# timestep.  Pairs further apart than this are set to NaN.
MAX_TIME_OFFSET_DAYS = 3.0

# ── Visual style ──────────────────────────────────────────────────────────────
MODEL_COLOR = "#0173b2"   # seaborn colorblind blue
OBS_COLOR = "#de8f05"     # seaborn colorblind orange
BAND_ALPHA = 0.22

# Wadden Sea map extent [lon_min, lon_max, lat_min, lat_max]
MAP_EXTENT_LONLAT = (4.65, 6.65, 52.85, 53.6)

## 1. Schema probe

Prints the layout of both CSVs: separator, columns, row counts, and — for the
RWS export — every distinct parameter, unit, `hoedanigheid` and station code,
plus the magnitude of any coordinate columns.

**Read this output before running anything else.** The
`(parameter, hoedanigheid, eenheid)` combinations listed here are exactly what
the unit conversion in the next section is keyed on; anything not covered there
raises rather than silently passing through.

In [38]:
# ──────────────────────────────────────────────────────────────────────────────
# SCHEMA PROBE  —  run this first
# ──────────────────────────────────────────────────────────────────────────────
# No existing script in this repo reads Chemistry_data_via_Waterinfo_RWS.csv, so
# its layout is confirmed here rather than assumed.  Read the printout before
# trusting anything downstream: the parameter / eenheid / hoedanigheid
# combinations listed below are exactly what the unit conversion is keyed on.

_DESCRIPTOR_HINTS = (
    "parameter", "grootheid", "eenheid", "hoedanigheid", "locatie", "limiet",
    "kwaliteit", "compartiment", "meetpunt", "waardebepaling",
)


def sniff_delimiter(path: Path) -> str:
    """Guess the field separator from the header line."""
    with open(path, "r", encoding="utf-8", errors="replace") as fh:
        sample = fh.readline() + fh.readline()
    try:
        return csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
    except csv.Error:
        # Fall back on whichever candidate appears most often in the header.
        counts = {d: sample.count(d) for d in [";", ",", "\t", "|"]}
        return max(counts, key=counts.get)


def probe_csv(name: str, path: Path, n_preview: int = 5) -> pd.DataFrame:
    """Print the layout of one observation CSV and return it as raw strings."""
    print("=" * 78)
    print(f"{name}  —  {path}")
    print("=" * 78)
    if not path.exists():
        print("  [ERROR] file not found")
        return pd.DataFrame()

    sep = sniff_delimiter(path)
    print(f"detected separator: {sep!r}")

    # dtype=str keeps Dutch decimal commas and leading zeros intact; every
    # numeric column is cast explicitly further down.
    df = pd.read_csv(path, sep=sep, dtype=str, na_values=["NA", ""],
                     encoding="utf-8", encoding_errors="replace",
                     engine="python")
    print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print("\ncolumns:")
    for c in df.columns:
        n_uniq = df[c].nunique(dropna=True)
        example = df[c].dropna().iloc[0] if df[c].notna().any() else "—"
        print(f"  {c:<38} nunique={n_uniq:<8} e.g. {str(example)[:40]}")

    print("\nfirst rows:")
    with pd.option_context("display.max_columns", None, "display.width", 250):
        print(df.head(n_preview))

    # Descriptor columns: these drive the parameter/unit mapping.
    desc_cols = [c for c in df.columns
                 if any(h in c.lower() for h in _DESCRIPTOR_HINTS)]
    for c in desc_cols:
        vc = df[c].value_counts(dropna=False)
        if len(vc) > 60:
            print(f"\n{c}: {len(vc)} distinct values (showing top 30)")
            print(vc.head(30).to_string())
        else:
            print(f"\n{c}:")
            print(vc.to_string())

    # Anything that could be a coordinate — the magnitude tells us the CRS.
    for c in df.columns:
        if c.lower().split(".")[-1] in ("x", "y", "lon", "lat", "longitude", "latitude"):
            vals = pd.to_numeric(df[c].str.replace(",", ".", regex=False), errors="coerce")
            if vals.notna().any():
                print(f"\ncoordinate column {c}: min={vals.min():.4f} max={vals.max():.4f}")

    return df


_probe_frames = {name: probe_csv(name, path) for name, path in OBS_CSV.items()}

NIOZ_JETTY  —  /export/lv9/projects/dws/results/validation/nutrients/Jetty_HWseries.csv
detected separator: ','
shape: 2,122 rows x 19 columns

columns:
  SampleID                               nunique=2122     e.g. HW7401
  timestamp                              nunique=2122     e.g. 1974-02-07 07:15:00
  T                                      nunique=248      e.g. 5
  S                                      nunique=224      e.g. 29.96
  SecchiDepth                            nunique=62       e.g. 0.3
  TN                                     nunique=821      e.g. 55.38
  TP                                     nunique=531      e.g. 1.448
  PO4                                    nunique=912      e.g. 2.8
  NO3                                    nunique=1505     e.g. 52.2
  NO2                                    nunique=972      e.g. 2.8
  NH4                                    nunique=1438     e.g. 8.046
  DON                                    nunique=586      e.g. 8.48
  DOP           

## 2. Units

Target units: **mmol m⁻³** for the molar species (identical to µmol L⁻¹) and
**mg m⁻³** for chlorophyll (identical to µg L⁻¹).

The one genuinely dangerous conversion is the mass basis. RWS reports the same
substance both as element mass and as whole-ion mass, and the two differ by a
large factor:

| reported as | → mmol m⁻³ per mg L⁻¹ | ratio to element basis |
|---|---|---|
| mg **N** L⁻¹ | 71.39 | — |
| mg **NO₃** L⁻¹ | 16.13 | 4.43 |
| mg **P** L⁻¹ | 32.29 | — |
| mg **PO₄** L⁻¹ | 10.53 | 3.07 |
| mg **Si** L⁻¹ | 35.61 | — |
| mg **SiO₂** L⁻¹ | 16.64 | 2.14 |

`unit_factor()` therefore resolves the basis from the `hoedanigheid` field when
it names an element and falls back on the ion implied by the parameter code
otherwise — and raises a `ValueError` on anything else rather than defaulting to
1.0, which would produce plausible-looking but wrong figures.

The NIOZ Jetty nutrients are µmol L⁻¹ and `Chl` is mg m⁻³ (metadata block in
`pelagic_validation.ipynb` cell 2 and the NIOZ Dataverse series,
doi:10.25850/nioz/7b.b.5j), so they need no scaling. The unit-audit cell in
section 5 re-checks that assumption against the model distributions.

In [39]:

# ──────────────────────────────────────────────────────────────────────────────
# UNIT CONVERSION
# ──────────────────────────────────────────────────────────────────────────────
# Target units:  molar species -> mmol m-3  (identical to umol L-1)
#                Chla          -> mg m-3    (identical to ug L-1)
#
# RWS reports the same substance both as element mass and as compound mass.
# mg NO3/L and mg N/L differ by a factor 4.43, mg PO4/L and mg P/L by 3.07;
# picking the wrong one produces figures that look plausible and are wrong.
# The conversion is therefore keyed on the (parameter, hoedanigheid, eenheid)
# triplet and *raises* on anything it does not recognise.

MOLAR_MASS = {
    # elements / molecules used as the mass basis
    "N": 14.0067, "P": 30.973762, "Si": 28.0855, "O2": 31.9988, "C": 12.011,
    # whole ions, used when hoedanigheid does not name an element
    "NO3": 62.0049, "NO2": 46.0055, "NH4": 18.0385,
    "PO4": 94.9714, "SiO2": 60.0843,
}

# Which whole molecule a parameter code refers to, used only when the
# hoedanigheid field does not already state the mass basis (N / P / Si / O2).
PARAM_COMPOUND = {
    "NO3": "NO3", "NO2": "NO2", "NH4": "NH4",
    "NO3NO2": "NO3", "NO2NO3": "NO3",   # combined NOx, mass-basis fallback
    "PO4": "PO4", "SIO2": "SiO2", "SI": "SiO2", "O2": "O2",
}

# hoedanigheid_code variants seen in the RWS Waterinfo export (Nnf/Npg/Pnf/...
# = dissolved/particulate-bound fractions expressed as the given element).
HOED_BASIS = {
    "N": "N", "NNF": "N", "NPG": "N",
    "P": "P", "PNF": "P", "PG": "P",
    "SI": "Si", "SINF": "Si",
    "O2": "O2",
    "C": "C", "CNF": "C", "CPG": "C", "CDG": "C",
}

# Volume conversion to grams per litre.
_MASS_PER_LITRE = {
    "g/l": 1.0, "mg/l": 1e-3, "ug/l": 1e-6, "ng/l": 1e-9,
    "g/m3": 1e-3, "mg/m3": 1e-6, "ug/m3": 1e-9,
    "mg/dm3": 1e-3, "ug/dm3": 1e-6,
}
# Already-molar units, expressed as a factor to mmol m-3.
_MOLAR_TO_MMOL_M3 = {
    "umol/l": 1.0, "umol/dm3": 1.0, "mmol/m3": 1.0, "umol/kg": 1.0,
    "mmol/l": 1e3, "mol/m3": 1e3, "mmol/dm3": 1e3,
    "nmol/l": 1e-3, "umol/m3": 1e-3,
}
# Chlorophyll is a mass concentration, not molar.
_CHL_TO_MG_M3 = {
    "ug/l": 1.0, "mg/m3": 1.0, "ug/dm3": 1.0,
    "mg/l": 1e3, "g/m3": 1e3, "ng/l": 1e-3, "ug/m3": 1e-3,
}

# RWS parameter code -> model variable.  Codes are upper-cased and stripped of
# separators before lookup, so "NO3-N", "no3_n" and "NO3" all match "NO3N"/"NO3".
# EXTEND THIS after reading the schema-probe printout.
RWS_PARAM_MAP = {
    "NO3": "N3n", "NO3N": "N3n", "NITRAAT": "N3n",
    "NO3NO2": "N3n", "NO2NO3": "N3n",   # combined NOx reported as nitrate
    "NH4": "N4n", "NH4N": "N4n", "AMMONIUM": "N4n",
    "PO4": "N1p", "PO4P": "N1p", "FOSFAAT": "N1p", "OPGELOSTORTHOFOSFAAT": "N1p",
    "SIO2": "N5s", "SI": "N5s", "SILICAAT": "N5s",
    "O2": "O2o", "ZUURSTOF": "O2o",
    "CHLFA": "Chla", "CHLFAA": "Chla", "CHLA": "Chla", "CHLOROFYLA": "Chla",
}
# Deliberately NOT mapped – total nutrient pools are not comparable to the
# model's dissolved inorganic pools:  Ntot, Ptot, Kj (Kjeldahl), NO2 alone.
RWS_PARAM_IGNORE = {"NTOT", "PTOT", "NKJ", "KJN", "NO2", "PO4TOT", "NTOTAAL", "PTOTAAL"}


def _norm_code(value) -> str:
    """Upper-case a code and strip spaces, dashes, underscores and dots."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    return re.sub(r"[\s\-_.]", "", str(value)).upper()


def _norm_unit(value) -> str:
    """Normalise a unit string: lower case, ASCII mu, no spaces, no superscripts."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    u = str(value).strip().lower()
    u = u.replace("µ", "u").replace("μ", "u")   # micro sign / greek mu
    u = u.replace("³", "3").replace("^3", "3").replace("**3", "3")
    u = u.replace(" ", "")
    # "mg/l N" style: the mass basis is handled separately, drop the trailing tag
    u = re.sub(r"(n|p|si|o2|c)$", "", u) if re.match(r"^[a-z]+/[a-z0-9]+(n|p|si|o2|c)$", u) else u
    return u


def unit_factor(model_var: str, param_code: str, hoedanigheid: str, eenheid: str) -> float:
    """
    Factor by which an observed value must be multiplied to reach the model unit
    (mmol m-3, or mg m-3 for Chla).

    Raises ValueError on any combination that is not explicitly covered — a
    silent fallback to 1.0 here is the most dangerous failure mode in the whole
    notebook, so it is not offered.
    """
    unit = _norm_unit(eenheid)
    hoed = _norm_code(hoedanigheid)
    param = _norm_code(param_code)

    if model_var == "Chla":
        if unit in _CHL_TO_MG_M3:
            return _CHL_TO_MG_M3[unit]
        raise ValueError(
            f"Unrecognised chlorophyll unit {eenheid!r} (normalised {unit!r}). "
            f"Add it to _CHL_TO_MG_M3."
        )

    if unit in _MOLAR_TO_MMOL_M3:
        return _MOLAR_TO_MMOL_M3[unit]

    if unit not in _MASS_PER_LITRE:
        raise ValueError(
            f"Unrecognised unit {eenheid!r} (normalised {unit!r}) for parameter "
            f"{param_code!r}. Add it to _MASS_PER_LITRE or _MOLAR_TO_MMOL_M3."
        )

    # Mass basis: hoedanigheid wins when it names an element, otherwise the
    # value is the mass of the whole ion implied by the parameter code.
    basis = HOED_BASIS.get(hoed)
    if basis is None:
        basis = PARAM_COMPOUND.get(param)
    if basis is None:
        raise ValueError(
            f"Cannot determine the mass basis for parameter {param_code!r} with "
            f"hoedanigheid {hoedanigheid!r} and unit {eenheid!r}. Either add the "
            f"parameter to PARAM_COMPOUND or map the hoedanigheid code."
        )

    # value [unit] -> g/L -> mol/L -> umol/L == mmol/m3
    return _MASS_PER_LITRE[unit] / MOLAR_MASS[basis] * 1e6


# ── NIOZ Jetty ────────────────────────────────────────────────────────────────
# Metadata documented in output_scripts/pelagic_validation.ipynb (cell 2) and in
# the NIOZ Dataverse series (doi:10.25850/nioz/7b.b.5j):
#   TSM mg/L | Chl mg/m3 | TOC, POC, DOC mgC/L | NO3, NH4, PO4 umol/L
# umol/L is numerically identical to mmol/m3, so the nutrients need no scaling.
# The unit-audit cell below re-checks this against the model percentiles, which
# is what would expose an unexpected unit as an order-of-magnitude offset.
JETTY_COL_MAP = {
    "Chl": "Chla",
    "NO3": "N3n",
    "NH4": "N4n",
    "PO4": "N1p",
    "SiO2": "N5s",   # present in some releases of the series; skipped if absent
    "Si": "N5s",
    "O2": "O2o",
}
JETTY_FACTOR = {
    "Chla": 1.0,   # mg m-3   -> mg m-3
    "N3n": 1.0,    # umol L-1 -> mmol m-3
    "N4n": 1.0,
    "N1p": 1.0,
    "N5s": 1.0,
    "O2o": 1.0,
}

print("Unit conversion self-test (value 1.0 -> mmol m-3):")
for _p, _h, _u in [("NO3", "N", "mg/l"), ("NO3", "NVT", "mg/l"),
                   ("PO4", "P", "mg/l"), ("PO4", "NVT", "mg/l"),
                   ("NH4", "N", "ug/l"), ("SiO2", "Si", "mg/l"),
                   ("O2", "NVT", "mg/l"), ("NO3", "N", "umol/l")]:
    _mv = RWS_PARAM_MAP[_norm_code(_p)]
    print(f"  {_p:<5} hoedanigheid={_h:<4} {_u:<7} -> x {unit_factor(_mv, _p, _h, _u):9.4f}  ({_mv})")
print(f"  CHLFA hoedanigheid=NVT  ug/l    -> x {unit_factor('Chla', 'CHLFA', 'NVT', 'ug/l'):9.4f}  (Chla)")


Unit conversion self-test (value 1.0 -> mmol m-3):
  NO3   hoedanigheid=N    mg/l    -> x   71.3944  (N3n)
  NO3   hoedanigheid=NVT  mg/l    -> x   16.1278  (N3n)
  PO4   hoedanigheid=P    mg/l    -> x   32.2854  (N1p)
  PO4   hoedanigheid=NVT  mg/l    -> x   10.5295  (N1p)
  NH4   hoedanigheid=N    ug/l    -> x    0.0714  (N4n)
  SiO2  hoedanigheid=Si   mg/l    -> x   35.6056  (N5s)
  O2    hoedanigheid=NVT  mg/l    -> x   31.2512  (O2o)
  NO3   hoedanigheid=N    umol/l  -> x    1.0000  (N3n)
  CHLFA hoedanigheid=NVT  ug/l    -> x    1.0000  (Chla)


## 3. Observation loaders

`load_rws()` auto-detects which of the two RWS export schemas the file uses —
the `;`-separated uppercase DONAR download (`LOCATIE_CODE`, `WAARNEMINGDATUM`,
`NUMERIEKEWAARDE`, `LAT`/`LON`), parsed the same way as in
`pelagic_validation.ipynb` cell 7, or the `,`-separated Waterinfo/ddlpy export
(`locatie.code`, `tijdstip`, `numeriekewaarde`, `geom` WKT), as in cell 13.
Everything is read as strings and cast explicitly, which survives Dutch decimal
commas.

Handled along the way: the DONAR `999999999` missing code, `<` detection-limit
flags (kept as LOD/2 by default and carried as a `below_lod` flag so those
points can be drawn hollow), the `OW` surface-water compartment filter, and
station coordinates in WGS84, RD (EPSG:28992) or UTM31N (EPSG:25831) — the CRS
is detected from the magnitude of the values and printed.

`load_jetty()` reshapes the fixed-station series into the same column layout so
both sources share one downstream code path.

In [40]:
# ──────────────────────────────────────────────────────────────────────────────
# OBSERVATION LOADERS
# ──────────────────────────────────────────────────────────────────────────────

# Canonical name -> candidate column names, lower-cased.  Two RWS export
# schemas are in circulation and both are handled:
#   * DONAR/Waterinfo download  : ';'-separated, UPPERCASE, LAT/LON, dd-mm-yyyy
#     (the schema parsed in pelagic_validation.ipynb cell 7)
#   * Waterinfo API / ddlpy     : ','-separated, dotted lower case, ISO tijdstip
#     (the schema parsed in pelagic_validation.ipynb cell 13)
RWS_COL_CANDIDATES = {
    "station":   ["locatie_code", "locatie.code", "locatiecode", "meetpunt.identificatie",
                  "locatie.naam", "locatie_naam", "meetpuntidentificatie",
                  "code", "naam"],
    "datetime":  ["tijdstip", "datumtijd", "datum_tijd"],
    "date":      ["waarnemingdatum", "datum", "waarneming_datum"],
    "time":      ["waarnemingtijd", "tijd", "waarneming_tijd"],
    "value":     ["numeriekewaarde", "waarde", "meetwaarde"],
    "parameter": ["parameter_code", "parameter.code", "parameter"],
    "parameter_desc": ["parameter_wat_omschrijving", "parameter_omschrijving"],
    "quantity":  ["grootheid_code", "grootheid.code", "grootheid"],
    "unit":      ["eenheid_code", "eenheid.code", "eenheid"],
    "basis":     ["hoedanigheid_code", "hoedanigheid.code", "hoedanigheid"],
    "limit":     ["limietsymbool", "limiet_symbool", "limietsymbool.code"],
    "lon":       ["lon", "longitude", "geografischepunt.x"],
    "lat":       ["lat", "latitude", "geografischepunt.y"],
    "x":         ["x", "locatie.x", "geometriepunt.x", "xcoordinaat"],
    "y":         ["y", "locatie.y", "geometriepunt.y", "ycoordinaat"],
    "geom":      ["geom", "geometrie", "wkt"],
    "compartment": ["compartiment_code", "compartiment.code", "compartiment"],
}


def _resolve_columns(df: pd.DataFrame) -> dict:
    """Map canonical names onto the columns actually present in *df*."""
    lower = {c.lower().strip(): c for c in df.columns}
    found = {}
    for canon, candidates in RWS_COL_CANDIDATES.items():
        for cand in candidates:
            if cand in lower:
                found[canon] = lower[cand]
                break
    return found


def _to_float(series: pd.Series) -> pd.Series:
    """Cast a string column to float, tolerating Dutch decimal commas."""
    return pd.to_numeric(
        series.astype(str).str.strip().str.replace(",", ".", regex=False),
        errors="coerce",
    )


def _classify_rws_param(desc: str) -> str:
    """
    Map a Dutch parameter_wat_omschrijving string onto a RWS_PARAM_MAP key.

    Needed because this export's parameter_code/grootheid_code columns are
    always "NVT" — the substance name only appears in this free-text field.
    Returns "" for anything not relevant to validation (e.g. totals, Kjeldahl).
    """
    d = str(desc).lower()
    if "ammonium" in d:
        return "NH4"
    if "som nitraat en nitriet" in d:
        return "NO3NO2"
    if "nitraat" in d and "nitriet" not in d:
        return "NO3"
    if "nitriet" in d and "nitraat" not in d:
        return "NO2"
    if "fosfaat" in d:
        return "PO4"
    if "silicaat" in d or "silicium" in d:
        return "SIO2"
    if "chlorofyl" in d:
        return "CHLFA"
    if "zuurstof" in d:
        return "O2"
    return ""


def parse_geom(geom_str):
    """Parse 'POINT (lon lat)' -> (lon, lat).  From pelagic_validation.ipynb."""
    if isinstance(geom_str, str) and geom_str.upper().startswith("POINT"):
        coords = geom_str[geom_str.index("(") + 1: geom_str.index(")")].split()
        return float(coords[0]), float(coords[1])
    return np.nan, np.nan


def to_wgs84(x: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray, str]:
    """
    Convert projected station coordinates to lon/lat, detecting the CRS from the
    magnitude of the values.  No precedent exists in this repo, so the detected
    CRS is printed rather than assumed silently.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    xm, ym = np.nanmax(np.abs(x)), np.nanmax(np.abs(y))

    if xm <= 180.0 and ym <= 90.0:
        return x, y, "EPSG:4326 (already lon/lat)"
    if xm < 3.5e5 and 3.0e5 < ym < 7.0e5:
        src = "EPSG:28992"          # Rijksdriehoek
    elif 3.0e5 < xm < 9.0e5 and ym > 5.0e6:
        src = "EPSG:25831"          # UTM zone 31N, what ddlpy returns
    else:
        raise ValueError(
            f"Cannot identify the CRS of the station coordinates "
            f"(max |x| = {xm:.1f}, max |y| = {ym:.1f}). Set it explicitly."
        )
    if not _HAS_PYPROJ:
        raise RuntimeError(f"Coordinates look like {src} but pyproj is unavailable.")
    tf = Transformer.from_crs(src, "EPSG:4326", always_xy=True)
    lon, lat = tf.transform(x, y)
    return np.asarray(lon), np.asarray(lat), src


def load_rws() -> pd.DataFrame:
    """
    Read the RWS chemistry export and return one row per (station, timestamp,
    model_var) with the value converted to the model's unit.

    Columns: station, lon, lat, timestamp, model_var, obs_value, below_lod.
    """
    path = OBS_CSV["RWS"]
    sep = sniff_delimiter(path)
    raw = pd.read_csv(path, sep=sep, dtype=str, na_values=["NA", ""],
                      encoding="utf-8", encoding_errors="replace", engine="python")
    cols = _resolve_columns(raw)
    print(f"RWS: {len(raw):,} raw rows, separator {sep!r}")
    print(f"     resolved columns: {cols}")

    missing = [k for k in ("station", "value", "unit") if k not in cols]
    if missing:
        raise KeyError(
            f"Could not find RWS column(s) {missing} in {list(raw.columns)}. "
            f"Extend RWS_COL_CANDIDATES."
        )

    df = pd.DataFrame(index=raw.index)
    df["station"] = raw[cols["station"]].astype(str).str.strip()

    # ── timestamp ─────────────────────────────────────────────────────────────
    if "datetime" in cols:
        ts = pd.to_datetime(raw[cols["datetime"]], errors="coerce", utc=True)
        df["timestamp"] = ts.dt.tz_convert(None)
    elif "date" in cols:
        combined = raw[cols["date"]].astype(str)
        if "time" in cols:
            combined = combined + " " + raw[cols["time"]].astype(str)
        df["timestamp"] = pd.to_datetime(combined, dayfirst=True, errors="coerce")
    else:
        raise KeyError("No date/time column found in the RWS export.")

    # ── substance, unit, mass basis ───────────────────────────────────────────
    # parameter.code names the substance; grootheid.code names the quantity
    # (CONCTTE etc.).  Prefer the parameter, fall back on the quantity.
    # Some RWS exports leave parameter_code == "NVT" for every row; the
    # substance then only appears in the free-text parameter_wat_omschrijving.
    if "parameter_desc" in cols:
        df["param_raw"] = raw[cols["parameter_desc"]].map(_classify_rws_param)
    else:
        param_col = cols.get("parameter") or cols.get("quantity")
        df["param_raw"] = raw[param_col].astype(str).str.strip()
    df["unit_raw"] = raw[cols["unit"]].astype(str).str.strip()
    df["basis_raw"] = raw[cols["basis"]].astype(str).str.strip() if "basis" in cols else ""

    # ── value, missing codes, detection limits ────────────────────────────────
    df["raw_value"] = _to_float(raw[cols["value"]])
    n0 = len(df)
    df = df[np.isfinite(df["raw_value"])]
    df = df[df["raw_value"] != RWS_MISSING_SENTINEL]
    df = df[df["raw_value"] < RWS_MISSING_SENTINEL / 1e3]   # any 9.99e8-style code
    print(f"     dropped {n0 - len(df):,} rows with missing / sentinel values")

    if "limit" in cols:
        lim = raw.loc[df.index, cols["limit"]].astype(str).str.strip()
        df["below_lod"] = lim.eq("<")
    else:
        df["below_lod"] = False
    n_lod = int(df["below_lod"].sum())
    if n_lod:
        if DROP_BELOW_DETECTION:
            df = df[~df["below_lod"]]
            print(f"     dropped {n_lod:,} below-detection-limit rows")
        else:
            df.loc[df["below_lod"], "raw_value"] *= 0.5
            print(f"     {n_lod:,} below-detection-limit rows substituted with LOD/2")

    # ── surface water only, when the export says which compartment ────────────
    if "compartment" in cols:
        comp = raw.loc[df.index, cols["compartment"]].astype(str).str.strip().str.upper()
        keep = comp.isin(["OW", "OPPERVLAKTEWATER", ""]) | comp.isna()
        if (~keep).any():
            print(f"     dropped {int((~keep).sum()):,} rows outside compartment OW")
            df = df[keep]

    # ── map parameters onto model variables ───────────────────────────────────
    codes = df["param_raw"].map(_norm_code)
    df["model_var"] = codes.map(RWS_PARAM_MAP)
    unmapped = sorted(set(codes[df["model_var"].isna()]) - RWS_PARAM_IGNORE - {""})
    if unmapped:
        print(f"     [INFO] parameters present but not mapped: {unmapped}")
        print(f"            add them to RWS_PARAM_MAP if they should be validated")
    df = df[df["model_var"].notna()].copy()

    # ── unit conversion ───────────────────────────────────────────────────────
    combos = df[["model_var", "param_raw", "basis_raw", "unit_raw"]].drop_duplicates()
    factors = {}
    print("     unit conversions applied:")
    for _, r in combos.iterrows():
        key = (r["model_var"], r["param_raw"], r["basis_raw"], r["unit_raw"])
        factors[key] = unit_factor(r["model_var"], r["param_raw"], r["basis_raw"], r["unit_raw"])
        print(f"       {r['param_raw']:<10} [{r['unit_raw']:<8}] hoedanigheid="
              f"{r['basis_raw'] or '-':<5} -> {r['model_var']:<5} x {factors[key]:.5g}")

    keys = list(zip(df["model_var"], df["param_raw"], df["basis_raw"], df["unit_raw"]))
    df["obs_value"] = df["raw_value"].to_numpy() * np.array([factors[k] for k in keys])

    # ── station coordinates ───────────────────────────────────────────────────
    if "lon" in cols and "lat" in cols:
        lon = _to_float(raw.loc[df.index, cols["lon"]])
        lat = _to_float(raw.loc[df.index, cols["lat"]])
        df["lon"], df["lat"] = lon.to_numpy(), lat.to_numpy()
        print("     coordinates: LAT/LON columns, EPSG:4326")
    elif "geom" in cols:
        xy = raw.loc[df.index, cols["geom"]].map(parse_geom)
        df["lon"] = [p[0] for p in xy]
        df["lat"] = [p[1] for p in xy]
        print("     coordinates: parsed from WKT geom, EPSG:4326")
    elif "x" in cols and "y" in cols:
        lon, lat, src = to_wgs84(_to_float(raw.loc[df.index, cols["x"]]).to_numpy(),
                                 _to_float(raw.loc[df.index, cols["y"]]).to_numpy())
        df["lon"], df["lat"] = lon, lat
        print(f"     coordinates: x/y columns interpreted as {src}")
    else:
        raise KeyError("No station coordinates found in the RWS export.")

    df = df.dropna(subset=["timestamp", "lon", "lat", "obs_value"])
    out = df[["station", "lon", "lat", "timestamp", "model_var", "obs_value", "below_lod"]]
    out = out.sort_values(["station", "timestamp"]).reset_index(drop=True)
    print(f"     {len(out):,} usable rows, "
          f"{out['station'].nunique()} stations, "
          f"{out['timestamp'].dt.year.min()}–{out['timestamp'].dt.year.max()}")
    return out


def rws_to_wide(obs_long: pd.DataFrame) -> pd.DataFrame:
    """
    One row per water sample (station + timestamp), one column per model
    variable.  DIN and DIN:DIP are only defined where both species were
    measured in the same sample.
    """
    keys = ["station", "lon", "lat", "timestamp"]
    wide = (obs_long
            .pivot_table(index=keys, columns="model_var",
                         values="obs_value", aggfunc="mean")
            .reset_index())
    wide.columns.name = None

    lod = (obs_long
           .pivot_table(index=keys, columns="model_var",
                        values="below_lod", aggfunc="max")
           .reset_index())
    lod.columns.name = None
    lod = lod.rename(columns={v: f"{v}_below_lod" for v in obs_long["model_var"].unique()})
    wide = wide.merge(lod, on=keys, how="left")

    for v in VALIDATION_VARS:
        if v not in wide.columns:
            wide[v] = np.nan
        flag = f"{v}_below_lod"
        wide[flag] = wide[flag].fillna(False).astype(bool) if flag in wide.columns else False
    return add_derived(wide)


def add_derived(df: pd.DataFrame, suffix: str = "") -> pd.DataFrame:
    """Add DIN and DIN:DIP to a table holding {N3n,N4n,N1p}{suffix} columns."""
    n3, n4, n1 = f"N3n{suffix}", f"N4n{suffix}", f"N1p{suffix}"
    if n3 in df.columns and n4 in df.columns:
        df[f"DIN{suffix}"] = df[n3] + df[n4]
    else:
        df[f"DIN{suffix}"] = np.nan
    if f"DIN{suffix}" in df.columns and n1 in df.columns:
        dip = df[n1].where(df[n1] > 0.01)      # guard division by ~zero phosphate
        df[f"DIN_DIP{suffix}"] = df[f"DIN{suffix}"] / dip
    else:
        df[f"DIN_DIP{suffix}"] = np.nan
    return df


def load_jetty() -> pd.DataFrame:
    """
    NIOZ jetty high-water series -> one row per sampling time, one column per
    model variable, in model units.  Follows pelagic_validation.ipynb cell 24.
    """
    path = OBS_CSV["NIOZ_JETTY"]
    raw = pd.read_csv(path, na_values=["NA", ""], parse_dates=["timestamp"])
    print(f"Jetty: {len(raw):,} raw rows, columns {list(raw.columns)}")

    raw = raw.dropna(subset=["timestamp"])
    out = pd.DataFrame({"timestamp": raw["timestamp"]})
    for obs_col, mvar in JETTY_COL_MAP.items():
        if obs_col not in raw.columns:
            continue
        if mvar in out.columns and out[mvar].notna().any():
            continue        # first matching alias wins (e.g. SiO2 before Si)
        out[mvar] = pd.to_numeric(raw[obs_col], errors="coerce") * JETTY_FACTOR[mvar]
        print(f"       {obs_col:<6} -> {mvar:<5} x {JETTY_FACTOR[mvar]}")
    for v in VALIDATION_VARS:
        if v not in out.columns:
            out[v] = np.nan
            print(f"       [INFO] no observed column for {v}; panel will be empty")

    out = add_derived(out)
    full_range = (out["timestamp"].min(), out["timestamp"].max())
    out = out[out["timestamp"].dt.year == MODEL_YEAR].sort_values("timestamp")
    out = out.reset_index(drop=True)
    print(f"       series covers {full_range[0].date()} – {full_range[1].date()}; "
          f"{len(out)} samples kept for {MODEL_YEAR}")
    return out

## 4. Model loading and collocation

Ported from `benthic_validation_spinup.ipynb`, with one substantive change: the
nutrient fields are 4-D `(time, level, yc, xc)`, so **the surface layer is
selected before `.load()`** — the full 4-D block for six variables is ~6 GB,
the surface slice ~0.6 GB.

The grid is curvilinear GETM: 1-D `xc`/`yc` in metres with 2-D auxiliary
`lonc(yc,xc)` / `latc(yc,xc)`, which is why collocation goes through a KD-tree
rather than a coordinate lookup. The tree is restricted to **wet** cells
(`bathymetry > -10`), so a station on a tidal flat snaps to the nearest wet
neighbour instead of returning a fill value.

Consecutive monthly files repeat the timestamp at the month boundary, so
duplicate times are removed both within and across files.

Nearest-time matching always returns *something*, so a sample taken in a month
the run does not cover would otherwise pair with whatever timestep happens to be
closest. Pairs more than `MAX_TIME_OFFSET_DAYS` apart therefore have their model
value voided, and the count is reported.

In [41]:
# ──────────────────────────────────────────────────────────────────────────────
# MODEL LOADING AND COLLOCATION
# ──────────────────────────────────────────────────────────────────────────────
# Ported from output_scripts/benthic_validation_spinup.ipynb.  The one
# substantive change: the nutrient fields are 4-D (time, level, yc, xc), so the
# surface layer is selected *before* .load() — the full 4-D block for six
# variables would be ~6 GB, the surface slice is ~0.6 GB.

EARTH_R_KM = 6371.0088


def build_kdtree(lon2d: np.ndarray, lat2d: np.ndarray, bathy2d: np.ndarray):
    """
    cKDTree of valid, *wet* model cell centres.

    Land cells carry bathymetry == BATHY_FILL (-10); excluding them means a
    station sitting on a tidal flat just outside the wet domain snaps to the
    nearest wet neighbour instead of returning a fill value.
    """
    mask = (
        (lon2d > LON_VALID[0]) & (lon2d < LON_VALID[1])
        & (lat2d > LAT_VALID[0]) & (lat2d < LAT_VALID[1])
        & (bathy2d > BATHY_FILL)
    )
    flat_iy, flat_ix = np.where(mask)
    tree = cKDTree(np.column_stack([lon2d[mask], lat2d[mask]]))
    return tree, flat_iy, flat_ix


def load_grid(spinup_dir: Path):
    """Static grid fields; identical in every file of the run."""
    any_nc = next(iter(sorted(spinup_dir.glob("dws_500m.3d.*.nc"))), None)
    if any_nc is None:
        raise FileNotFoundError(f"No NetCDF files in {spinup_dir}")
    with xr.open_dataset(any_nc, mask_and_scale=False) as ds:
        lon2d = ds["lonc"].values.copy()
        lat2d = ds["latc"].values.copy()
        bathy2d = ds["bathymetry"].values.copy()
    return lon2d, lat2d, bathy2d


def load_model_surface_year(spinup_dir: Path, year: int) -> xr.Dataset | None:
    """
    Open every monthly file for *year*, keep only the surface layer and the
    variables to be validated, and return one deduplicated dataset.

    Consecutive monthly files repeat the timestamp at the month boundary, so
    duplicates are removed both within and across files.
    """
    nc_files = sorted(spinup_dir.glob(NC_PATTERN.format(year=year)))
    if not nc_files:
        return None

    datasets = []
    for fp in nc_files:
        with xr.open_dataset(fp, mask_and_scale=False) as dsi:
            present = [v for v in VALIDATION_VARS if v in dsi.data_vars]
            missing = [v for v in VALIDATION_VARS if v not in dsi.data_vars]
            if missing:
                warnings.warn(f"{fp.name}: variables {missing} absent, skipped.")
            if not present:
                continue
            sub = dsi[present]
            if "level" in sub.dims:
                sub = sub.isel(level=SURFACE_LEVEL)
            sub = sub.load()

        t_idx = pd.DatetimeIndex(pd.to_datetime(sub["time"].values))
        if t_idx.has_duplicates:
            sub = sub.isel(time=~t_idx.duplicated())
        datasets.append(sub)

    if not datasets:
        return None

    ds = xr.concat(datasets, dim="time", data_vars="minimal",
                   coords="minimal", compat="override", join="override")
    ds = ds.sortby("time")
    t_idx = pd.DatetimeIndex(pd.to_datetime(ds["time"].values))
    if t_idx.has_duplicates:
        ds = ds.isel(time=~t_idx.duplicated())

    # Fill values (-9999) -> NaN.  A -900 threshold is used rather than the
    # benthic notebook's -1, because BFM can legitimately produce small
    # negative concentrations that are worth seeing rather than hiding.
    for v in ds.data_vars:
        ds[v] = ds[v].where(ds[v] > DATA_FILL_THRESHOLD)

    print(f"  loaded {len(nc_files)} files -> {ds.sizes['time']} unique timesteps "
          f"({pd.to_datetime(ds['time'].values[0]).date()} – "
          f"{pd.to_datetime(ds['time'].values[-1]).date()}), "
          f"variables {list(ds.data_vars)}")
    return ds


def nearest_time_idx(model_times: pd.DatetimeIndex, obs_times: np.ndarray) -> np.ndarray:
    """Index into *model_times* of the nearest timestamp for each observation."""
    mt = model_times.to_numpy(dtype="datetime64[ns]")
    ot = np.asarray(obs_times, dtype="datetime64[ns]")
    right = np.searchsorted(mt, ot, side="left")
    left = np.clip(right - 1, 0, len(mt) - 1)
    right = np.clip(right, 0, len(mt) - 1)
    left_diff = np.abs(ot - mt[left]).astype(np.int64)
    right_diff = np.abs(mt[right] - ot).astype(np.int64)
    return np.where(right_diff < left_diff, right, left)


def haversine_km(lon1, lat1, lon2, lat2) -> np.ndarray:
    """Great-circle distance in km; used to report how far a station snapped."""
    lon1, lat1, lon2, lat2 = map(np.radians, (lon1, lat1, lon2, lat2))
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * EARTH_R_KM * np.arcsin(np.sqrt(a))


def scatter_metrics(x_obs: np.ndarray, y_mod: np.ndarray) -> dict:
    """n, bias, RMSE and Pearson r for a pair of arrays (from the benthic nb)."""
    x_obs = np.asarray(x_obs, dtype=float)
    y_mod = np.asarray(y_mod, dtype=float)
    valid = np.isfinite(x_obs) & np.isfinite(y_mod)
    x, y = x_obs[valid], y_mod[valid]
    if len(x) < 3:
        return {"n": len(x), "bias": np.nan, "rmse": np.nan, "r": np.nan}
    r, _ = pearsonr(x, y)
    return {
        "n": len(x),
        "bias": float(np.mean(y - x)),
        "rmse": float(np.sqrt(np.mean((y - x) ** 2))),
        "r": float(r),
    }


# ── RWS: nearest wet cell, nearest timestep ───────────────────────────────────

def collocate_rws(ds: xr.Dataset, obs: pd.DataFrame, tree, flat_iy, flat_ix,
                  lon2d, lat2d) -> pd.DataFrame:
    """Append a `<var>_model` column for every validated variable."""
    obs = obs.copy()
    _, nn = tree.query(np.column_stack([obs["lon"].to_numpy(), obs["lat"].to_numpy()]), k=1)
    obs["iy"] = flat_iy[nn].astype(int)
    obs["ix"] = flat_ix[nn].astype(int)
    obs["cell_lon"] = lon2d[obs["iy"], obs["ix"]]
    obs["cell_lat"] = lat2d[obs["iy"], obs["ix"]]
    obs["snap_km"] = haversine_km(obs["lon"], obs["lat"], obs["cell_lon"], obs["cell_lat"])

    far = obs["snap_km"] > MAX_SNAP_DISTANCE_KM
    if far.any():
        print(f"  [WARN] dropping {int(far.sum()):,} rows from "
              f"{obs.loc[far, 'station'].nunique()} station(s) snapping "
              f">{MAX_SNAP_DISTANCE_KM} km: "
              f"{sorted(obs.loc[far, 'station'].unique())[:10]}")
        obs = obs[~far].copy()

    model_times = pd.DatetimeIndex(pd.to_datetime(ds["time"].values))
    it = nearest_time_idx(model_times, obs["timestamp"].to_numpy(dtype="datetime64[ns]"))
    obs["model_time"] = model_times[it]
    obs["time_offset_h"] = (obs["model_time"] - obs["timestamp"]).dt.total_seconds() / 3600.0

    it_xr = xr.DataArray(it.astype(int), dims="obs")
    iy_xr = xr.DataArray(obs["iy"].to_numpy().astype(int), dims="obs")
    ix_xr = xr.DataArray(obs["ix"].to_numpy().astype(int), dims="obs")

    for v in VALIDATION_VARS:
        if v not in ds.data_vars:
            obs[f"{v}_model"] = np.nan
            continue
        obs[f"{v}_model"] = ds[v].isel(time=it_xr, yc=iy_xr, xc=ix_xr).values

    obs = _blank_stale_matches(obs, VALIDATION_VARS, "RWS")
    obs = add_derived(obs, suffix="_model")
    return obs.reset_index(drop=True)


def _blank_stale_matches(df: pd.DataFrame, variables, label: str) -> pd.DataFrame:
    """
    Void model values matched to a timestep more than MAX_TIME_OFFSET_DAYS away.

    Nearest-time matching always returns *something*; without this guard a
    sample taken in a month the run does not cover would silently pair with the
    closest timestep the run does have.
    """
    stale = df["time_offset_h"].abs() > MAX_TIME_OFFSET_DAYS * 24.0
    if stale.any():
        print(f"  [WARN] {label}: {int(stale.sum()):,} of {len(df):,} samples are "
              f">{MAX_TIME_OFFSET_DAYS:g} days from any model timestep — "
              f"model values voided (max offset {df['time_offset_h'].abs().max() / 24:.0f} d)")
        for v in variables:
            col = f"{v}_model"
            if col in df.columns:
                df.loc[stale, col] = np.nan
    return df


# ── NIOZ jetty: one fixed station, full-year series ───────────────────────────

def extract_jetty_series(ds: xr.Dataset, tree, flat_iy, flat_ix,
                         lon2d, lat2d, bathy2d) -> pd.DataFrame:
    """
    Surface time series at the jetty: the nearest wet cell, plus the mean, min
    and max over the surrounding (2*JETTY_BOX_HALF+1)^2 block of wet cells,
    which is drawn as an uncertainty band in the seasonal-cycle figure.
    """
    _, nn = tree.query([[JETTY_LON, JETTY_LAT]], k=1)
    iy, ix = int(flat_iy[nn[0]]), int(flat_ix[nn[0]])
    d = haversine_km(JETTY_LON, JETTY_LAT, lon2d[iy, ix], lat2d[iy, ix])
    print(f"  jetty ({JETTY_LON}, {JETTY_LAT}) -> cell (yc={iy}, xc={ix}) at "
          f"({lon2d[iy, ix]:.4f}, {lat2d[iy, ix]:.4f}), {d * 1000:.0f} m away, "
          f"depth {bathy2d[iy, ix]:.2f} m")
    if d > 1.0:
        print(f"  [WARN] snap distance {d:.2f} km exceeds one 500 m cell diagonal — "
              f"check JETTY_LON / JETTY_LAT against the dataset metadata.")

    h = JETTY_BOX_HALF
    y0, y1 = max(iy - h, 0), min(iy + h + 1, lon2d.shape[0])
    x0, x1 = max(ix - h, 0), min(ix + h + 1, lon2d.shape[1])
    wet = xr.DataArray(bathy2d[y0:y1, x0:x1] > BATHY_FILL, dims=("yc", "xc"))
    n_wet = int(wet.sum())

    out = pd.DataFrame({"time": pd.to_datetime(ds["time"].values)})
    for v in VALIDATION_VARS:
        if v not in ds.data_vars:
            out[v] = np.nan
            out[f"{v}_lo"] = np.nan
            out[f"{v}_hi"] = np.nan
            continue
        out[v] = ds[v].isel(yc=iy, xc=ix).values
        block = ds[v].isel(yc=slice(y0, y1), xc=slice(x0, x1)).where(wet)
        out[f"{v}_lo"] = block.min(dim=("yc", "xc"), skipna=True).values
        out[f"{v}_hi"] = block.max(dim=("yc", "xc"), skipna=True).values
    print(f"  {n_wet} wet cells in the {y1 - y0}x{x1 - x0} neighbourhood band")

    out = add_derived(out)
    # The band for the derived variables is taken from the derived block ends.
    out["DIN_lo"] = out["N3n_lo"] + out["N4n_lo"]
    out["DIN_hi"] = out["N3n_hi"] + out["N4n_hi"]
    out["DIN_DIP_lo"] = out["DIN_lo"] / out["N1p_hi"].where(out["N1p_hi"] > 0.01)
    out["DIN_DIP_hi"] = out["DIN_hi"] / out["N1p_lo"].where(out["N1p_lo"] > 0.01)
    return out


def pair_jetty(model_ts: pd.DataFrame, obs: pd.DataFrame) -> pd.DataFrame:
    """Match each jetty sample to the nearest model timestep."""
    model_times = pd.DatetimeIndex(model_ts["time"])
    it = nearest_time_idx(model_times, obs["timestamp"].to_numpy(dtype="datetime64[ns]"))
    paired = obs.copy()
    paired["model_time"] = model_times[it]
    paired["time_offset_h"] = (
        paired["model_time"] - paired["timestamp"]).dt.total_seconds() / 3600.0
    for v in ALL_VARS:
        paired[f"{v}_model"] = model_ts[v].to_numpy()[it]
    return _blank_stale_matches(paired, ALL_VARS, "NIOZ_JETTY")

## 5. Unit audit

The distributions side by side. Look for order-of-magnitude agreement, not for
skill — that comes later. A median ratio near **4.43**, **3.07** or **2.14**
between an observation source and the model points at a nitrogen/nitrate,
phosphorus/phosphate or silicon/silica mass-basis mix-up rather than at a model
bias.

In [42]:
# ──────────────────────────────────────────────────────────────────────────────
# UNIT AUDIT
# ──────────────────────────────────────────────────────────────────────────────
# The cheapest way to catch a wrong unit conversion is to look at the
# distributions side by side.  A 4.43x offset on nitrate or 3.07x on phosphate
# is a nitrogen/nitrate or phosphorus/phosphate mass-basis mix-up, not a model
# bias.  The "plausible" column holds the colour limits used in the BFM plotting
# script by J. van der Molen and is a rough order-of-magnitude guide only.

def unit_audit(jetty_obs: pd.DataFrame | None,
               rws_obs: pd.DataFrame | None,
               model_ts: pd.DataFrame | None) -> pd.DataFrame:
    rows = []
    for v in ALL_VARS:
        lo, hi = PLAUSIBLE_RANGE.get(v, (np.nan, np.nan))
        for label, series in (
            ("model (jetty cell)", None if model_ts is None else model_ts.get(v)),
            ("obs NIOZ_JETTY", None if jetty_obs is None else jetty_obs.get(v)),
            ("obs RWS", None if rws_obs is None else rws_obs.get(v)),
        ):
            if series is None:
                continue
            s = pd.to_numeric(series, errors="coerce").dropna()
            if s.empty:
                rows.append({"variable": v, "unit": VAR_UNITS[v], "source": label,
                             "n": 0, "p5": np.nan, "median": np.nan, "p95": np.nan,
                             "plausible": f"{lo:g}–{hi:g}"})
                continue
            rows.append({
                "variable": v, "unit": VAR_UNITS[v], "source": label, "n": len(s),
                "p5": float(np.percentile(s, 5)),
                "median": float(np.median(s)),
                "p95": float(np.percentile(s, 95)),
                "plausible": f"{lo:g}–{hi:g}",
            })
    audit = pd.DataFrame(rows)

    # Ratio of observed to modelled median — the number to eyeball.
    med = audit.pivot_table(index="variable", columns="source", values="median")
    if "model (jetty cell)" in med.columns:
        for src in [c for c in med.columns if c.startswith("obs")]:
            med[f"{src} / model"] = med[src] / med["model (jetty cell)"]

    with pd.option_context("display.width", 200, "display.max_columns", None,
                           "display.float_format", lambda x: f"{x:,.3f}"):
        print(audit.to_string(index=False))
        print("\nmedian ratios (a value near 4.43, 3.07 or 2.14 suggests a "
              "N/NO3, P/PO4 or Si/SiO2 mass-basis error):")
        print(med.to_string())
    return audit

## 6. Figures

Style, metrics box and PDF-at-300-dpi saving conventions follow
`benthic_validation_spinup.ipynb`. Everything lands in `OUT_DIR`
(`…/validation/nutrients/no_BP1c/`).

In [43]:
# ──────────────────────────────────────────────────────────────────────────────
# PLOTTING — shared style and helpers
# ──────────────────────────────────────────────────────────────────────────────

def _apply_theme() -> None:
    """Publication theme, identical to benthic_validation_spinup.ipynb."""
    sns.set_theme(style="ticks")
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "black",
        "axes.linewidth": 1.0,
        "figure.facecolor": "white",
        "savefig.facecolor": "white",
    })


def _metrics_box(ax, metrics: dict, color: str, loc: str = "lower right") -> None:
    """Monospace n / bias / RMSE / r box, as used in the benthic notebook."""
    lines = [
        f"$n$  = {metrics['n']:,}",
        f"bias = {metrics['bias']:+.3g}",
        f"RMSE = {metrics['rmse']:.3g}",
        f"$r$   = {metrics['r']:.2f}",
    ]
    va, ha = ("bottom", "right") if loc == "lower right" else ("top", "left")
    x, y = (0.97, 0.04) if loc == "lower right" else (0.03, 0.97)
    ax.text(x, y, "\n".join(lines), transform=ax.transAxes, va=va, ha=ha,
            size="small", linespacing=1.55, family="monospace",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec=color,
                      lw=1.2, alpha=0.88), zorder=5)


def _axis_limit(*arrays) -> float:
    """Shared upper limit for a 1:1 scatter: the 99th percentile, padded."""
    vals = np.concatenate([np.asarray(a, dtype=float).ravel() for a in arrays])
    vals = vals[np.isfinite(vals) & (vals >= 0)]
    return float(np.percentile(vals, 99)) * 1.08 if vals.size else 1.0


def _save(fig, filename: str) -> None:
    out = OUT_DIR / filename
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


def _grid_shape(n: int) -> tuple[int, int]:
    """Rows, cols for n panels, preferring 2 rows."""
    ncol = int(np.ceil(n / 2))
    return 2, ncol

### 6a. NIOZ Jetty

The band around the model line is the range across the 3×3 block of wet cells
around the jetty cell — a rough measure of how sensitive the comparison is to
the exact cell chosen at a 500 m coastal grid point.

Remember that these are **high-water** samples matched against roughly daily
instantaneous model output; the tidal phase is not resolved, so treat a constant
offset here as a sampling artefact until shown otherwise.

In [44]:
# ──────────────────────────────────────────────────────────────────────────────
# PLOTTING — NIOZ jetty (fixed station, seasonal cycle)
# ──────────────────────────────────────────────────────────────────────────────

def plot_jetty_timeseries(model_ts: pd.DataFrame, obs: pd.DataFrame) -> None:
    """Model surface series at the jetty cell (line + neighbourhood band) vs obs."""
    _apply_theme()
    nrow, ncol = _grid_shape(len(ALL_VARS))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.4 * nrow),
                             sharex=True, constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()

    for ax, v in zip(axes, ALL_VARS):
        has_model = v in model_ts.columns and model_ts[v].notna().any()
        if has_model:
            ax.plot(model_ts["time"], model_ts[v], lw=1.4, color=MODEL_COLOR,
                    label=f"model (cell)", zorder=3)
            lo, hi = f"{v}_lo", f"{v}_hi"
            if lo in model_ts.columns and model_ts[lo].notna().any():
                ax.fill_between(model_ts["time"], model_ts[lo], model_ts[hi],
                                color=MODEL_COLOR, alpha=BAND_ALPHA, lw=0,
                                label=f"model ({2 * JETTY_BOX_HALF + 1}x{2 * JETTY_BOX_HALF + 1} range)",
                                zorder=2)

        o = obs[["timestamp", v]].dropna() if v in obs.columns else pd.DataFrame()
        if not o.empty:
            ax.scatter(o["timestamp"], o[v], s=24, color=OBS_COLOR,
                       edgecolors="#7a4d00", linewidths=0.4, alpha=0.9,
                       label="obs (NIOZ jetty, HW)", zorder=4)

        if not has_model and o.empty:
            ax.text(0.5, 0.5, "no data", ha="center", va="center",
                    transform=ax.transAxes, color="#888888")
        ax.set_title(VAR_LABEL[v], pad=4)
        ax.set_ylabel(VAR_UNITS[v])
        ax.grid(True, alpha=0.25)
        ax.margins(x=0.01)

    for ax in axes[len(ALL_VARS):]:
        ax.set_axis_off()

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(labels),
               framealpha=0.9, edgecolor="#cccccc", bbox_to_anchor=(0.5, -0.04))
    fig.suptitle(f"NIOZ jetty (Marsdiep) — surface, {MODEL_YEAR}",
                 fontweight="bold", size="large")
    fig.autofmt_xdate()
    _save(fig, f"jetty_timeseries_{MODEL_YEAR}.pdf")


def plot_jetty_scatter(paired: pd.DataFrame) -> None:
    """Model vs observed at the jetty, one panel per variable."""
    _apply_theme()
    nrow, ncol = _grid_shape(len(ALL_VARS))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.0 * ncol, 4.0 * nrow),
                             constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()

    for ax, v in zip(axes, ALL_VARS):
        x = pd.to_numeric(paired.get(v), errors="coerce").to_numpy() \
            if v in paired.columns else np.array([])
        y = pd.to_numeric(paired.get(f"{v}_model"), errors="coerce").to_numpy() \
            if f"{v}_model" in paired.columns else np.array([])
        good = np.isfinite(x) & np.isfinite(y) if x.size and y.size else np.array([], bool)
        if good.sum() < 1:
            ax.text(0.5, 0.5, "no paired data", ha="center", va="center",
                    transform=ax.transAxes, color="#888888")
            ax.set_title(VAR_LABEL[v], pad=4)
            continue

        gmax = _axis_limit(x[good], y[good])
        ax.scatter(x[good], y[good], s=26, alpha=0.7, linewidths=0.4,
                   color=MODEL_COLOR, edgecolors="white", zorder=3)
        ax.plot([0, gmax], [0, gmax], color="#666666", lw=1.0, ls="--",
                zorder=1, label="1:1")
        if good.sum() > 3:
            slope, intercept, *_ = linregress(x[good], y[good])
            xf = np.array([0.0, gmax])
            ax.plot(xf, slope * xf + intercept, color=OBS_COLOR, lw=1.8, zorder=2,
                    label="OLS")
        ax.set_xlim(0, gmax)
        ax.set_ylim(0, gmax)
        _metrics_box(ax, scatter_metrics(x[good], y[good]), MODEL_COLOR)
        ax.set_title(VAR_LABEL[v], pad=4)
        ax.set_xlabel(f"observed [{VAR_UNITS[v]}]")
        ax.set_ylabel(f"modelled [{VAR_UNITS[v]}]")
        ax.xaxis.set_major_locator(mticker.MaxNLocator(4, prune="both"))
        ax.yaxis.set_major_locator(mticker.MaxNLocator(4, prune="both"))

    for ax in axes[len(ALL_VARS):]:
        ax.set_axis_off()

    fig.suptitle(f"NIOZ jetty — model vs observed, {MODEL_YEAR}",
                 fontweight="bold", size="large")
    _save(fig, f"jetty_scatter_{MODEL_YEAR}.pdf")

### 6b. RWS chemistry

Points are coloured by station; hollow markers are values reported below the
detection limit (substituted with LOD/2). The maps show the mean model − obs
per station over 2015.

In [45]:
# ──────────────────────────────────────────────────────────────────────────────
# PLOTTING — RWS chemistry (many stations)
# ──────────────────────────────────────────────────────────────────────────────

def _station_palette(stations) -> dict:
    stations = list(stations)
    colors = sns.color_palette("husl", max(len(stations), 1)).as_hex()
    return dict(zip(stations, colors))


def plot_rws_scatter(coll: pd.DataFrame, n_legend: int = 12) -> None:
    """One file per variable: model vs observed, coloured by station."""
    _apply_theme()
    order = (coll["station"].value_counts().index.tolist())
    palette = _station_palette(order)

    for v in ALL_VARS:
        ocol, mcol = v, f"{v}_model"
        if ocol not in coll.columns or mcol not in coll.columns:
            continue
        sub = coll[[ "station", ocol, mcol]].copy()
        lodcol = f"{v}_below_lod"
        sub["below_lod"] = coll[lodcol] if lodcol in coll.columns else False
        sub = sub[np.isfinite(sub[ocol]) & np.isfinite(sub[mcol])]
        if len(sub) < 3:
            print(f"  [INFO] {v}: only {len(sub)} paired RWS values, scatter skipped.")
            continue

        gmax = _axis_limit(sub[ocol], sub[mcol])
        fig, ax = plt.subplots(figsize=(6.2, 5.6), constrained_layout=True)

        for st in order:
            s = sub[sub["station"] == st]
            if s.empty:
                continue
            solid = s[~s["below_lod"]]
            hollow = s[s["below_lod"]]
            if not solid.empty:
                ax.scatter(solid[ocol], solid[mcol], s=22, alpha=0.75,
                           linewidths=0, color=palette[st], zorder=3,
                           rasterized=True, label=st)
            if not hollow.empty:
                ax.scatter(hollow[ocol], hollow[mcol], s=26, alpha=0.75,
                           facecolors="none", linewidths=0.9,
                           edgecolors=palette[st], zorder=3, rasterized=True)

        ax.plot([0, gmax], [0, gmax], color="#666666", lw=1.0, ls="--",
                zorder=1, label="1:1")
        x = sub[ocol].to_numpy()
        y = sub[mcol].to_numpy()
        if len(sub) > 3:
            slope, intercept, *_ = linregress(x, y)
            xf = np.array([0.0, gmax])
            ax.plot(xf, slope * xf + intercept, color="#333333", lw=1.8,
                    zorder=2, label="OLS")

        ax.set_xlim(0, gmax)
        ax.set_ylim(0, gmax)
        _metrics_box(ax, scatter_metrics(x, y), MODEL_COLOR)
        ax.set_xlabel(f"observed [{VAR_UNITS[v]}]")
        ax.set_ylabel(f"modelled [{VAR_UNITS[v]}]")
        ax.set_title(f"RWS — {VAR_LABEL[v]}, {MODEL_YEAR}", fontweight="bold")

        handles, labels = ax.get_legend_handles_labels()
        keep = [(h, l) for h, l in zip(handles, labels)][:n_legend]
        if keep:
            hs, ls = zip(*keep)
            ax.legend(hs, ls, loc="upper left", fontsize=7, ncol=2,
                      framealpha=0.9, edgecolor="#cccccc")
        if sub["below_lod"].any():
            ax.text(0.02, 0.02, "hollow = below detection limit (LOD/2)",
                    transform=ax.transAxes, fontsize=7, color="#555555")

        _save(fig, f"rws_scatter_{v}_{MODEL_YEAR}.pdf")


def plot_rws_station_cycles(coll: pd.DataFrame, n_stations: int = 6) -> None:
    """Seasonal cycle at the best-sampled stations, one file per variable."""
    _apply_theme()

    for v in ALL_VARS:
        ocol, mcol = v, f"{v}_model"
        if ocol not in coll.columns or mcol not in coll.columns:
            continue
        sub = coll[np.isfinite(coll[ocol]) & np.isfinite(coll[mcol])]
        if sub.empty:
            continue
        top = sub["station"].value_counts().head(n_stations).index.tolist()
        if not top:
            continue

        ncol = min(3, len(top))
        nrow = int(np.ceil(len(top) / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.2 * nrow),
                                 sharex=True, constrained_layout=True)
        axes = np.atleast_1d(axes).ravel()

        for ax, st in zip(axes, top):
            s = sub[sub["station"] == st].sort_values("timestamp")
            ax.plot(s["model_time"], s[mcol], "-o", ms=4, lw=1.2,
                    color=MODEL_COLOR, label="model", zorder=3)
            ax.scatter(s["timestamp"], s[ocol], s=28, color=OBS_COLOR,
                       edgecolors="#7a4d00", linewidths=0.4, label="obs", zorder=4)
            m = scatter_metrics(s[ocol].to_numpy(), s[mcol].to_numpy())
            ax.set_title(f"{st}  (n={m['n']})", pad=4, size="medium")
            ax.set_ylabel(VAR_UNITS[v])
            ax.grid(True, alpha=0.25)

        for ax in axes[len(top):]:
            ax.set_axis_off()

        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", ncol=2,
                   framealpha=0.9, edgecolor="#cccccc", bbox_to_anchor=(0.5, -0.05))
        fig.suptitle(f"RWS station seasonal cycles — {VAR_LABEL[v]}, {MODEL_YEAR}",
                     fontweight="bold", size="large")
        fig.autofmt_xdate()
        _save(fig, f"rws_station_cycles_{v}_{MODEL_YEAR}.pdf")


def plot_rws_error_maps(coll: pd.DataFrame) -> None:
    """Mean model - obs per station, on a Wadden Sea map (one file per variable)."""
    if not _HAS_MAPS:
        print("  [SKIP] cartopy/contextily unavailable — no error maps.")
        return
    _apply_theme()
    proj = ccrs.Mercator()
    data_crs = ccrs.PlateCarree()

    for v in ALL_VARS:
        ocol, mcol = v, f"{v}_model"
        if ocol not in coll.columns or mcol not in coll.columns:
            continue
        sub = coll[np.isfinite(coll[ocol]) & np.isfinite(coll[mcol])].copy()
        if sub.empty:
            continue
        sub["error"] = sub[mcol] - sub[ocol]
        per_station = (sub.groupby("station")
                       .agg(lon=("lon", "mean"), lat=("lat", "mean"),
                            error=("error", "mean"), n=("error", "size"))
                       .reset_index())
        if per_station.empty:
            continue

        clim = float(np.nanpercentile(np.abs(per_station["error"]), 99))
        if not np.isfinite(clim) or clim == 0:
            clim = float(np.nanmax(np.abs(per_station["error"]))) or 1.0
        norm = mcolors.TwoSlopeNorm(vcenter=0, vmin=-clim, vmax=clim)

        fig, ax = plt.subplots(figsize=(8.5, 6.0),
                               subplot_kw={"projection": proj})
        ax.set_extent(MAP_EXTENT_LONLAT, crs=data_crs)
        try:
            ctx.add_basemap(ax, crs=proj.to_string(),
                            source=ctx.providers.Esri.WorldShadedRelief,
                            zoom="auto", zorder=1)
            for txt in ax.texts:
                txt.set_position([0.99, 0.01])
                txt.set_ha("right")
                txt.set_fontsize(5)
        except Exception:
            ax.set_facecolor("#cce5f0")

        sc = ax.scatter(per_station["lon"], per_station["lat"],
                        c=per_station["error"], cmap="RdBu_r", norm=norm,
                        s=70, marker="o", linewidths=0.6, edgecolors="#333333",
                        transform=data_crs, zorder=4)
        for _, r in per_station.iterrows():
            ax.text(r["lon"], r["lat"], f" {r['station']}", transform=data_crs,
                    fontsize=6, color="#222222", va="bottom", ha="left", zorder=5)

        gl = ax.gridlines(crs=data_crs, draw_labels=True, linewidth=0.35,
                          color="#888888", alpha=0.6, linestyle="--", zorder=2)
        gl.top_labels = False
        gl.right_labels = False
        gl.xlocator = mticker.FixedLocator([4.8, 5.2, 5.6, 6.0, 6.4])
        gl.ylocator = mticker.FixedLocator([52.9, 53.1, 53.3, 53.5])
        gl.xlabel_style = {"size": 9}
        gl.ylabel_style = {"size": 9}

        cb = fig.colorbar(sc, ax=ax, orientation="horizontal", extend="both",
                          pad=0.06, shrink=0.85)
        cb.set_label(f"mean model $-$ obs  [{VAR_UNITS[v]}]", fontsize=10)
        cb.ax.axvline(0, color="black", lw=1.2)

        ax.set_title(f"RWS — {VAR_LABEL[v]}, mean model error {MODEL_YEAR}",
                     fontweight="bold", pad=8)
        _save(fig, f"rws_error_map_{v}_{MODEL_YEAR}.pdf")

## 7. Run

Loads both observation sets, collocates them against `spinup_01`, prints the
time-axis and snap-distance diagnostics, writes the collocated tables and the
skill summary, and generates every figure. Each figure block is guarded so one
failing variable does not abort the run.

Sanity checks worth reading in the output:

* time axis ≈ 365–370 steps, monotonic, no duplicates;
* jetty snap distance below ~700 m (one 500 m cell diagonal);
* no station snapping more than `MAX_SNAP_DISTANCE_KM` away;
* few or no samples voided by the `MAX_TIME_OFFSET_DAYS` guard — a large count
  means the run does not cover the months the samples come from;
* non-zero matched-pair counts for every variable.

In [46]:
# ──────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ──────────────────────────────────────────────────────────────────────────────

def metrics_table(jetty_paired: pd.DataFrame | None,
                  rws_coll: pd.DataFrame | None) -> pd.DataFrame:
    rows = []
    for source, df in (("NIOZ_JETTY", jetty_paired), ("RWS", rws_coll)):
        if df is None or df.empty:
            continue
        for v in ALL_VARS:
            if v not in df.columns or f"{v}_model" not in df.columns:
                continue
            m = scatter_metrics(df[v].to_numpy(), df[f"{v}_model"].to_numpy())
            rows.append({"source": source, "variable": v, "unit": VAR_UNITS[v], **m})
    return pd.DataFrame(rows)


def _guard(label, fn, *args, **kwargs):
    """Run a figure block, reporting rather than aborting on failure."""
    try:
        fn(*args, **kwargs)
    except Exception as exc:
        warnings.warn(f"{label} failed: {type(exc).__name__}: {exc}")


def main() -> dict:
    results = {}

    # ── observations ──────────────────────────────────────────────────────────
    print("Loading observations …")
    jetty_obs = load_jetty()
    rws_long = load_rws()
    rws_obs = rws_to_wide(rws_long)
    rws_obs_year = rws_obs[rws_obs["timestamp"].dt.year == MODEL_YEAR].copy()
    print(f"RWS: {len(rws_obs):,} samples total, {len(rws_obs_year):,} in {MODEL_YEAR} "
          f"from {rws_obs_year['station'].nunique()} stations")
    if rws_obs_year.empty:
        warnings.warn(f"No RWS samples in {MODEL_YEAR}; only the jetty will be validated.")

    for spinup in SPINUP_NAMES:
        spinup_dir = BASE_OUTPUT_DIR / spinup
        print(f"\nProcessing {spinup} ({spinup_dir}) …")
        if not spinup_dir.exists():
            print(f"  [WARN] {spinup_dir} does not exist – skipping.")
            continue

        lon2d, lat2d, bathy2d = load_grid(spinup_dir)
        tree, flat_iy, flat_ix = build_kdtree(lon2d, lat2d, bathy2d)
        print(f"  grid {lon2d.shape}, {len(flat_iy):,} wet cells in the KD-tree")

        ds = load_model_surface_year(spinup_dir, MODEL_YEAR)
        if ds is None:
            print(f"  [WARN] no model files for {MODEL_YEAR} in {spinup_dir}.")
            continue

        mt = pd.DatetimeIndex(pd.to_datetime(ds["time"].values))
        print(f"  time axis: {len(mt)} steps, monotonic={mt.is_monotonic_increasing}, "
              f"duplicates={mt.has_duplicates}, "
              f"median step {pd.Series(mt).diff().median()}")

        # ── jetty ─────────────────────────────────────────────────────────────
        model_ts = extract_jetty_series(ds, tree, flat_iy, flat_ix,
                                        lon2d, lat2d, bathy2d)
        jetty_paired = pair_jetty(model_ts, jetty_obs)

        # ── RWS ───────────────────────────────────────────────────────────────
        if not rws_obs_year.empty:
            rws_coll = collocate_rws(ds, rws_obs_year, tree, flat_iy, flat_ix,
                                     lon2d, lat2d)
            print(f"  RWS collocated: {len(rws_coll):,} samples, median snap "
                  f"{rws_coll['snap_km'].median():.2f} km, median |Δt| "
                  f"{rws_coll['time_offset_h'].abs().median():.1f} h")
            for v in ALL_VARS:
                if f"{v}_model" in rws_coll.columns and v in rws_coll.columns:
                    n = int((rws_coll[v].notna() & rws_coll[f"{v}_model"].notna()).sum())
                    print(f"    {v:<8} {n:>6,} matched pairs")
        else:
            rws_coll = pd.DataFrame()

        ds.close()

        # ── unit audit ────────────────────────────────────────────────────────
        print("\nUnit audit — check these numbers before trusting the figures:")
        audit = unit_audit(jetty_obs, rws_coll if not rws_coll.empty else None, model_ts)

        # ── tables ────────────────────────────────────────────────────────────
        sp_dir = OUT_DIR / spinup
        sp_dir.mkdir(parents=True, exist_ok=True)
        model_ts.to_csv(sp_dir / f"jetty_model_surface_series_{MODEL_YEAR}.csv", index=False)
        jetty_paired.to_csv(sp_dir / f"jetty_collocated_model_vs_obs_{MODEL_YEAR}.csv", index=False)
        if not rws_coll.empty:
            rws_coll.to_csv(sp_dir / f"rws_collocated_model_vs_obs_{MODEL_YEAR}.csv", index=False)
        audit.to_csv(sp_dir / f"unit_audit_{MODEL_YEAR}.csv", index=False)

        metrics = metrics_table(jetty_paired, rws_coll if not rws_coll.empty else None)
        metrics.to_csv(OUT_DIR / f"nutrient_metrics_{MODEL_YEAR}.csv", index=False)
        print("\nSkill summary:")
        with pd.option_context("display.width", 200,
                               "display.float_format", lambda x: f"{x:,.3f}"):
            print(metrics.to_string(index=False))

        # ── figures ───────────────────────────────────────────────────────────
        print("\nGenerating figures …")
        _guard("jetty timeseries", plot_jetty_timeseries, model_ts, jetty_obs)
        _guard("jetty scatter", plot_jetty_scatter, jetty_paired)
        if not rws_coll.empty:
            _guard("RWS scatter", plot_rws_scatter, rws_coll)
            _guard("RWS station cycles", plot_rws_station_cycles, rws_coll)
            _guard("RWS error maps", plot_rws_error_maps, rws_coll)

        results[spinup] = {"model_ts": model_ts, "jetty": jetty_paired,
                           "rws": rws_coll, "metrics": metrics, "audit": audit}

    print(f"\nAll output written to: {OUT_DIR}")
    return results


results = main()

Loading observations …
Jetty: 2,122 raw rows, columns ['SampleID', 'timestamp', 'T', 'S', 'SecchiDepth', 'TN', 'TP', 'PO4', 'NO3', 'NO2', 'NH4', 'DON', 'DOP', 'Si', 'DIC', 'TSM', 'Chl', 'Kd_Secci', 'Kd_SPM']
       Chl    -> Chla  x 1.0
       NO3    -> N3n   x 1.0
       NH4    -> N4n   x 1.0
       PO4    -> N1p   x 1.0
       Si     -> N5s   x 1.0
       [INFO] no observed column for O2o; panel will be empty
       series covers 1974-02-07 – 2024-12-18; 40 samples kept for 2015
RWS: 49,266 raw rows, separator ','
     resolved columns: {'station': 'code', 'datetime': 'tijdstip', 'value': 'meetwaarde', 'parameter': 'parameter_code', 'parameter_desc': 'parameter_wat_omschrijving', 'quantity': 'grootheid_code', 'unit': 'eenheid_code', 'basis': 'hoedanigheid_code', 'limit': 'limietsymbool', 'x': 'x', 'y': 'y', 'geom': 'geom', 'compartment': 'compartiment_code'}
     dropped 3,223 rows with missing / sentinel values
     dropped 1,163 rows outside compartment OW
     unit conversions app